## Aggregation and Grouping

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

## Planets Data

In [21]:
planets = sns.load_dataset("planets")
planets.shape

(1035, 6)

In [22]:
planets.head()

,method,number,orbital_period,mass,distance,year
0,Radial Velocity,1,269.300,7.10,77.40,2006
1,Radial Velocity,1,874.774,2.21,56.95,2008
2,Radial Velocity,1,763.000,2.60,19.84,2011
3,Radial Velocity,1,326.030,19.40,110.62,2007
4,Radial Velocity,1,516.220,10.50,119.47,2009


In [23]:
planets.dropna().describe()

,number,orbital_period,mass,distance,year
count,498.00000,498.000000,498.000000,498.000000,498.000000
mean,1.73494,835.778671,2.509320,52.068213,2007.377510
std,1.17572,1469.128259,3.636274,46.596041,4.167284
min,1.00000,1.328300,0.003600,1.350000,1989.000000
25%,1.00000,38.272250,0.212500,24.497500,2005.000000
50%,1.00000,357.000000,1.245000,39.940000,2009.000000
75%,2.00000,999.600000,2.867500,59.332500,2011.000000
max,6.00000,17337.500000,25.000000,354.000000,2014.000000


## **Columns Indexing**

In [24]:
planets.groupby("method")

In [25]:
planets.groupby("method")["orbital_period"]

In [26]:
planets.groupby("method")["orbital_period"].median()

method
Astrometry                         631.180000
Eclipse Timing Variations         4343.500000
Imaging                          27500.000000
Microlensing                      3300.000000
Orbital Brightness Modulation        0.342887
Pulsar Timing                       66.541900
Pulsation Timing Variations       1170.000000
Radial Velocity                    360.200000
Transit                              5.714932
Transit Timing Variations           57.011000
Name: orbital_period, dtype: float64

### **Iteration over groups**

In [27]:
for (method, group) in planets.groupby("method"):
    print("{0:30s} shape={1}".format(method, group.shape))

Astrometry                     shape=(2, 6)
Eclipse Timing Variations      shape=(9, 6)
Imaging                        shape=(38, 6)
Microlensing                   shape=(23, 6)
Orbital Brightness Modulation  shape=(3, 6)
Pulsar Timing                  shape=(5, 6)
Pulsation Timing Variations    shape=(1, 6)
Radial Velocity                shape=(553, 6)
Transit                        shape=(397, 6)
Transit Timing Variations      shape=(4, 6)


### **Dispatch methods**

In [28]:
planets.groupby("method")["year"].describe().unstack()

       method                       
count  Astrometry                          2.0
       Eclipse Timing Variations           9.0
       Imaging                            38.0
       Microlensing                       23.0
       Orbital Brightness Modulation       3.0
                                         ...  
max    Pulsar Timing                    2011.0
       Pulsation Timing Variations      2007.0
       Radial Velocity                  2014.0
       Transit                          2014.0
       Transit Timing Variations        2014.0
Length: 80, dtype: float64

In [4]:
rng = np.random.RandomState(42)
ser = pd.Series(rng.rand(5))
ser

0    0.374540
1    0.950714
2    0.731994
3    0.598658
4    0.156019
dtype: float64

In [5]:
ser.sum()

np.float64(2.811925491708157)

In [6]:
ser.mean()

np.float64(0.5623850983416314)

In [8]:
df = pd.DataFrame({"A": rng.rand(5),
                   "B": rng.rand(5)})

df

,A,B
0,0.183405,0.611853
1,0.304242,0.139494
2,0.524756,0.292145
3,0.431945,0.366362
4,0.291229,0.456070


In [10]:
df.mean()

A    0.347115
B    0.373185
dtype: float64

In [11]:
df.mean(axis="columns")

0    0.397629
1    0.221868
2    0.408451
3    0.399153
4    0.373650
dtype: float64

## GroupBy: Split, Apply, Combine

In [13]:
df = pd.DataFrame({"key": ["A", "B", "C", "A", "B", "C"],
                   "data": range(6)}, columns=["key", "data"])

df

,key,data
0,A,0
1,B,1
2,C,2
3,A,3
4,B,4
5,C,5


In [14]:
df.groupby("key")

In [15]:
df.groupby("key").sum()

,data
key,
A,3
B,5
C,7


### **The GroupBy object**

### **Aggregate, filter, transform, apply**

In [16]:
rng = np.random.RandomState(0)
df = pd.DataFrame({"key": ["A", "B", "C", "A", "B", "C"],
                   "data1": range(6),
                   "data2": rng.randint(0, 10, 6)},
                   columns = ["key", "data1", "data2"])

df

,key,data1,data2
0,A,0,5
1,B,1,0
2,C,2,3
3,A,3,3
4,B,4,7
5,C,5,9


Aggregation

In [17]:
df.groupby("key").aggregate(["min", np.median, max])

C:\Users\Admin\AppData\Local\Temp\ipykernel_6264\3430934023.py:1: FutureWarning: The provided callable <function median at 0x0000020871DC1F80> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  df.groupby("key").aggregate(["min", np.median, max])
C:\Users\Admin\AppData\Local\Temp\ipykernel_6264\3430934023.py:1: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  df.groupby("key").aggregate(["min", np.median, max])


data1            data2           
      min median max   min median max
key                                  
A       0    1.5   3     3    4.0   5
B       1    2.5   4     0    3.5   7
C       2    3.5   5     3    6.0   9

In [18]:
df.groupby("key").aggregate({"data1": "min",
                             "data2": "max"})

,data1,data2
key,,
A,0,5
B,1,7
C,2,9


Filtering

In [19]:
def filter_func(x):
    return x["data2"].std() > 4

print(df); print(df.groupby("key").std());
print(df.groupby("key").filter(filter_func))

  key  data1  data2
0   A      0      5
1   B      1      0
2   C      2      3
3   A      3      3
4   B      4      7
5   C      5      9
       data1     data2
key                   
A    2.12132  1.414214
B    2.12132  4.949747
C    2.12132  4.242641
  key  data1  data2
1   B      1      0
2   C      2      3
4   B      4      7
5   C      5      9


Transformation

In [20]:
df.groupby("key").transform(lambda x:x - x.mean())

,data1,data2
0,-1.5,1.0
1,-1.5,-3.5
2,-1.5,-3.0
3,1.5,-1.0
4,1.5,3.5
5,1.5,3.0


The apply() method

In [29]:
def norm_by_data2(x):
    x["data1"] /= x["data2"].sum()
    return x

print(df); print(df.groupby("key").apply(norm_by_data2))

  key  data1  data2
0   A      0      5
1   B      1      0
2   C      2      3
3   A      3      3
4   B      4      7
5   C      5      9
      key     data1  data2
key                       
A   0   A  0.000000      5
    3   A  0.375000      3
B   1   B  0.142857      0
    4   B  0.571429      7
C   2   C  0.166667      3
    5   C  0.416667      9


C:\Users\Admin\AppData\Local\Temp\ipykernel_6264\2766167059.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df); print(df.groupby("key").apply(norm_by_data2))


### A list, array, series, or index providing the grouping keys

In [30]:
L = [0, 1, 0, 1, 2, 0]
print(df); print(df.groupby(L).sum())

  key  data1  data2
0   A      0      5
1   B      1      0
2   C      2      3
3   A      3      3
4   B      4      7
5   C      5      9
   key  data1  data2
0  ACC      7     17
1   BA      4      3
2    B      4      7


In [31]:
print(df); print(df.groupby(df["key"]).sum())

  key  data1  data2
0   A      0      5
1   B      1      0
2   C      2      3
3   A      3      3
4   B      4      7
5   C      5      9
     data1  data2
key              
A        3      8
B        5      7
C        7     12


#### A dictionary or series mapping index to group

In [32]:
df2 = df.set_index("key")
mapping = {"A": "vowel", "B": "consonant", "C": "consonant"}
print(df2); print(df2.groupby(mapping).sum())

     data1  data2
key              
A        0      5
B        1      0
C        2      3
A        3      3
B        4      7
C        5      9
           data1  data2
key                    
consonant     12     19
vowel          3      8


#### Grouping Example

In [33]:
decade = 10*(planets["year"] // 10)
decade = decade.astype(str) + "s"
decade.name = "decade"
planets.groupby(["method", decade])["number"].sum().unstack().fillna(0)

decade,1980s,1990s,2000s,2010s
method,,,,
Astrometry,0.0,0.0,0.0,2.0
Eclipse Timing Variations,0.0,0.0,5.0,10.0
Imaging,0.0,0.0,29.0,21.0
Microlensing,0.0,0.0,12.0,15.0
Orbital Brightness Modulation,0.0,0.0,0.0,5.0
Pulsar Timing,0.0,9.0,1.0,1.0
Pulsation Timing Variations,0.0,0.0,1.0,0.0
Radial Velocity,1.0,52.0,475.0,424.0
Transit,0.0,0.0,64.0,712.0
